## Objective

In this notebook we explore the possibility of deriving the sequence of dependencies in a chain or tree of events, i.e. that event C depends on event B which in turn depends on event A:

A -> B -> C

purely from the observed timestamps of those events.  We assume that a) events are correctly correlated, i.e. that event A1 is indeed related to B1 and C1, but importantly we do not know their sequence of dependency, b) the processes generating events are stochastic in nature, but yet c) there is a sufficient number of observed event chains/trees in order to determine statistically significant correlation, d) all event chains An -> Bn -> Cn etc. follow a common *super-set* of dependencies, in other words, an event chain may not include all possible events, however the sequence of dependencies between event chains do not otherwise contradict one another

## Terminology

Event set = the set of events of the same type, and that will appear in the same position in the tree structure, denoted by a capital letter: A, B, C
Event = one particular event in the event set, A1 being the first event in the event set A
An Event Chain/Tree = the dependency tree between the corresponding event in each set, i.e. A1 -> B1 -> C1
Depedency Tree = the generalised dependency tree that is true for all events in all event sets

## Simplifying Assumptions:
1. Timestamps are accurately recorded and synched.  There is no drift or correction needed on any of the event sets.
2. In order for a collection of events B to be dependent on a collation of events A, then the timestamp of every event in B must be later than the timestamp of every individual event in A, i.e. Bn >= An for all n items.  This constraint may need to be relaxed in practice to be some %age of timestamps (2 to 3 standard deviations?) in order to accomodate incorrectly recorded timestamps per #1 above.
3. It will be assumed that events will be dependent on other events whereever feasible.  Only if there is no likelihood of an event being depdendent on any another event will it be assumed to be a root node in the tree.
4. Where an event may succeed more than one other event, then the Pearson correlation will be used to determine which collection of events most closely correlate.  The Pearson correlation must be statistically significant, determined by a given pvalue threshold.

In [1]:
import numpy as np
from scipy import stats
from plotly import graph_objects as go
import ipycytoscape

In [2]:
"""
Toy data consisting of three event chains, both originating with event A
A -> B -> D
A -> C -> E -> F
A -> C -> E -> G
A -> H for 50% of events only

The processes generating events B, C will follow the same distribution.

The processes generating events D, E will similarly follow the same distribution.

Key assertions:
1) It is possible to determine that D suceeds B rather than C, i.e. Peason Rho is higher between D and B than D and C.
2) Similarly that E succeeds C rather than B
3) There is a single origin A

"""

NUMBER_OBSERVATIONS = 300   #Number of individual observations per collection of events

# Model the latencies of the individual events.
_A = np.absolute(stats.norm(loc=2, scale = 0.8).rvs(NUMBER_OBSERVATIONS).round(3))
_B = np.concatenate((np.absolute(stats.norm(loc=0.9, scale = 0.3).rvs(int(NUMBER_OBSERVATIONS / 2)).round(3))
                    ,np.absolute(stats.norm(loc=0.9, scale = 0.3).rvs(int(NUMBER_OBSERVATIONS / 2)).round(3)))
                    )
_C = np.absolute(stats.norm(loc=0.9, scale = 0.3).rvs(NUMBER_OBSERVATIONS).round(3))
_D = np.absolute(stats.norm(loc=2.5, scale = 1).rvs(NUMBER_OBSERVATIONS).round(3))
_E = np.absolute(stats.norm(loc=2.5, scale = 1).rvs(NUMBER_OBSERVATIONS).round(3))
_F = np.absolute(stats.norm(loc=0.9, scale = 0.2).rvs(NUMBER_OBSERVATIONS).round(3))
_G = np.absolute(stats.norm(loc=0.5, scale = 0.1).rvs(NUMBER_OBSERVATIONS).round(3))
_H = np.absolute(stats.norm(loc=1.5, scale = 0.6).rvs(NUMBER_OBSERVATIONS).round(3))

# Model the individual timestamps as the cumulative sum of latencies.  The result will be a 1D array of floating point numbers, assume these are the number of seconds since midnight
A = np.cumsum(_A)
B = A + _B
C = A + _C
D = B + _D
E = C + _E
F = E + _F
G = E + _G
H = A + _H

#But events in H should only be present 50% of the time
index = np.random.choice(a=(True, False), size=NUMBER_OBSERVATIONS, p=[0.5, 0.5])
H[index] = np.nan

#ToDo: events sets and lables should really be a dictionary...
event_sets = [A, B, C, D, E, F, G, H]
event_labels = ["A", "B", "C", "D", "E", "F", "G", "H"]
number_sets = len(event_sets)

In [3]:
fig = go.Figure()

for i, event_set in enumerate(event_sets):
    fig.add_trace(go.Scatter(x = list(range(len(event_set))),
                             y = event_set,
                            mode = "lines",
                            name = event_labels[i]

                        )
                )

fig.show()

In [ ]:
#Set a maximum pvalue for the Pearson correlation to be considered relevant
max_pval = 0.05

#Now we see if we can reverse-engineer the sequence of dependencies, purely from the timestamps of the event collections

#Create a matrix of all possible correlations, initialised to 0
#Rows correlate to the event, colums the dependency
corr = np.zeros((number_sets, number_sets))

for e in range(number_sets):
    for d in range(number_sets):

        #create an index of only those items which are not NAN in both the event and potential dependency.
        index = ~np.isnan(A) & ~np.isnan(H)

        #an event cannot be dependent on itself
        if e == d:  
            p = 0
        #test that the event is later than the potential dependency in all cases
        elif (event_sets[e][index] >= event_sets[d][index]).all():
            r = stats.pearsonr(event_sets[e][index], event_sets[d][index], alternative="greater")
            #the pearsonr function returns a tuple, [0] is the correlation co-efficient rho, [1] is the pvalue of the null hypothesis
            if r[1] < max_pval:     #if the pval is less than the max pval set above, then accept the pearson correlation
                p = r[0]
            else:
                p=0
        #events are not later than the potential dependency, therefore the correlation is zero
        else:
            p = 0
        #update the correlation matrix
        corr[e, d] = p

print(corr)

nodes = []
edges = []

#print the results in an understandable way
for e in range(number_sets):

    nodes.append({"data": {"id": event_labels[e], "label": event_labels[e]}})

    if (corr[e]==0).all():
        print("Event {e} is a root event".format(e=event_labels[e]))
    else:
        d = np.argmax(corr[e])

        index = ~np.isnan(A) & ~np.isnan(H)

        delta = (event_sets[e][index] - event_sets[d][index])
        mean_delta = np.round(np.mean(delta),3)
        std_delta = np.round(np.std(delta), 3)
        max_delta = np.round(np.max(delta), 3)
        min_delta = np.round(np.min(delta))

        label = "m={mean_delta}\n s={std_delta}\n max={max_delta}\n min={min_delta}".format(mean_delta=mean_delta, std_delta=std_delta, max_delta=max_delta, min_delta=min_delta)

        print("Event {e} is likely to be dependent on {d}".format(e=event_labels[e], d=event_labels[d]), label)

        edges.append({'data': {'source': event_labels[d], 'target': event_labels[e], "label": label,  "isdirected": True}})

"""
Expected Result:
A -> B -> D
A -> C -> E -> F
A -> C -> E -> 
A -> H
"""


[[0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.99999868 0.         0.         0.         0.         0.
  0.         0.        ]
 [0.99999846 0.         0.         0.         0.         0.
  0.         0.        ]
 [0.99998292 0.99998486 0.         0.         0.         0.
  0.         0.        ]
 [0.99998221 0.         0.99998375 0.         0.         0.
  0.         0.        ]
 [0.99998143 0.99997986 0.99998276 0.         0.9999995  0.
  0.         0.        ]
 [0.99998237 0.99998077 0.99998394 0.         0.99999986 0.
  0.         0.        ]
 [0.99999548 0.         0.         0.         0.         0.
  0.         0.        ]]
Event A is a root event
Event B is likely to be dependent on A m=0.914
 s=0.294
 max=1.742
 min=0.0
Event C is likely to be dependent on A m=0.888
 s=0.312
 max=1.793
 min=0.0
Event D is likely to be dependent on B m=2.502
 s=0.974
 max=4.549
 min=0.0
Event E is likely to be dependent on C m=2.554
 s=1.009
 max=5.113


'\nExpected Result:\nA -> B -> D\nA -> C -> E -> F\nA -> C -> E -> G\n'

In [6]:
#display the inferred dependency tree as a directed acylic graph
elements = {"nodes": nodes, 
            "edges": edges}

cytoscapeobj = ipycytoscape.CytoscapeWidget()
cytoscapeobj.graph.add_graph_from_json(elements, directed=True)
cytoscapeobj.set_layout(name="dagre")   #dagre, klay
cytoscapeobj.set_style([{
                          "selector":"node",
                          "style":{
                             "label-position": "right",
                             "font-family": "arial",
                             "font-size": "12px",
                             "label": "data(label)"
                          }
                       },
                       {
                          "selector":"edge",
                          "style":{
                             "width": 4,
                             "line-color": "#888",
                             "target-arrow-color": "#888",
                             "target-arrow-shape": "triangle",
                             "curve-style": "bezier",
                             "label": "data(label)",
                             "text-wrap": "wrap",
                             "font-family": "arial",
                             "font-size": "8px",
                          }
                       },
                       
                    ])


cytoscapeobj


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'label-pos…